In [ ]:
# =========================================
# 📦 1. Setup & Imports
# =========================================

%pip install pandas openpyxl

import pandas as pd


# =========================================
# 📂 2. Load Dataset
# =========================================

# Path to raw dataset
file_path = "../data/raw/online_retail_raw.csv"

# Read dataset
df = pd.read_csv(file_path, sep=';', encoding='utf-8')

# Initial preview
df.head()
df.info()


# =========================================
# 🔍 3. Initial Data Exploration
# =========================================

print("Columns:", df.columns.tolist())
print("\nShape:", df.shape)
print("\nMissing values per column:\n", df.isna().sum())


# =========================================
# 🧹 4. Data Cleaning
# =========================================

# --- Remove missing Customer IDs
if 'CustomerID' in df.columns:
    df = df.dropna(subset=['CustomerID'])

# --- Remove duplicates
df = df.drop_duplicates()

# --- Remove invalid values (negative Quantity / UnitPrice)
if 'Quantity' in df.columns and 'UnitPrice' in df.columns:
    df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# --- Standardize Country values
country_map = {
    'EIRE': 'Ireland',
    'RSA': 'South Africa',
    'Unspecified': 'Other',
    'European Community': 'Other'
}

df['Country'] = df['Country'].replace(country_map)

print("\nUnique Countries:\n", df['Country'].unique())

# --- Handle missing descriptions
df['Description'] = df['Description'].fillna('Unknown')

print("\nMissing values after cleaning:\n", df.isna().sum())


# =========================================
# 📅 5. Date Processing
# =========================================

# Convert to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Keep only full-year data (2011)
df['Year'] = df['InvoiceDate'].dt.year
df = df[df['Year'] >= 2011]
df = df.drop('Year', axis=1)

# Reset index after filtering
df = df.reset_index(drop=True)

# Create clean sequential index
df['index'] = range(len(df))


# =========================================
# 💾 6. Save Clean Dataset
# =========================================

df.to_csv(
    "../data/edited/online_retail_cleaned.csv",
    index=False
)

df.to_excel(
    "../data/edited/online_retail_cleaned.xlsx", index=False
    )

print("\nClean dataset saved.")
print("Final Shape:", df.shape)


# =========================================
# 📊 7. Exploratory Data Analysis (EDA)
# =========================================

df.describe()      # Summary statistics
df.nunique()       # Unique values per column
df.info()          # Final structure

df.head()